In [17]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("pedidos_limpios")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.mongodb.spark:mongo-spark-connector_2.12:10.7.0"
    )
    .config(
        "spark.mongodb.read.connection.uri",
        "mongodb://127.0.0.1:27017"
    )
    .config(
        "spark.mongodb.read.database",
        "tienda_etl"
    )
    .config(
        "spark.mongodb.read.collection",
        "pedidos"
    )
    .getOrCreate()
)

In [18]:
df = (spark.read.format("mongodb").load())
df.show()

+---+--------+--------------+---------------+--------------------+------------+--------+-----------------+
|_id|cantidad|     categoria|        cliente|               email|       fecha|  precio|         producto|
+---+--------+--------------+---------------+--------------------+------------+--------+-----------------+
|  1|       1|     DEPORTES |María Fernández|maría.fernández@g...|  24/01/2024|    NULL| Raqueta de Tenis|
|  2|       3|    TECNOLOGIA|María Fernández|maría.fernández@g...|Apr 17, 2024|3.400,99|      Monitor 27"|
|  3|       3|    TECNologia|María Fernández|maría.fernández@g...|  14/12/2024|    NULL| Teclado Mecánico|
|  4|       5|         hogar| Agustín Flores|agustín.flores@gm...|  2024-05-26|    NULL|  Espejo de Pared|
|  5|       1|        hogar |María Fernández|maría.fernández@g...|  2024-04-23|   42,00|  Espejo de Pared|
|  6|       1|         HOGAR| Agustín Flores|agustín.flores@gm...|  21/08/2024|  140,00|       Aspiradora|
|  7|       5|      deportes| Agustín

In [3]:
from pyspark.sql.functions import col, to_date, coalesce

df = df.withColumn(
    "fecha",
    coalesce(
        to_date(col("fecha"), "yyyy-MM-dd"),
        to_date(col("fecha"), "dd/MM/yyyy"),
        to_date(col("fecha"), "MMM dd, yyyy")
    )
)

df.select("fecha").show(truncate=False)

+----------+
|fecha     |
+----------+
|2024-01-24|
|2024-04-17|
|2024-12-14|
|2024-05-26|
|2024-04-23|
|2024-08-21|
|2024-09-24|
|2024-01-08|
|2024-07-21|
|2024-07-12|
|2024-11-06|
|2024-11-24|
|2024-09-25|
|2024-01-20|
|2024-12-16|
|2024-10-14|
|2024-02-08|
|2024-04-03|
|2024-10-16|
|2024-12-02|
+----------+
only showing top 20 rows



In [4]:
df = df.drop("_id")

antes = df.count()

df = df.dropDuplicates()

despues = df.count()

print("Antes:", antes)
print("Después:", despues)

Antes: 60
Después: 60


In [5]:
from pyspark.sql.functions import col, regexp_replace, when

df = df.withColumn(
    "precio",
    when(
        col("precio").contains(","),
        regexp_replace(
            regexp_replace(col("precio"), "\\.", ""),
            ",",
            "." 
        ).cast("double")
    ).otherwise(
        col("precio").cast("double")
    )
)

df = df.filter(
    col("precio").isNotNull() & (col("precio") > 0)
)

df.select("precio").show(truncate=False)

+-------+
|precio |
+-------+
|25.0   |
|28.0   |
|15.0   |
|890.0  |
|3400.99|
|48.0   |
|450.0  |
|450.0  |
|140.0  |
|42.0   |
|28.0   |
|42.0   |
|38.0   |
|45.0   |
|140.0  |
|1200.0 |
|89.9   |
|28.0   |
|65.0   |
|38.0   |
+-------+
only showing top 20 rows



In [6]:
mediana_cantidad = df.approxQuantile(
    "cantidad",
    [0.5],
    0.0
)[0]

df = df.fillna({
    "cantidad": int(mediana_cantidad)
})

df.show(truncate=False)

+--------+--------------+---------------+--------------------------+----------+-------+------------------+
|cantidad|categoria     |cliente        |email                     |fecha     |precio |producto          |
+--------+--------------+---------------+--------------------------+----------+-------+------------------+
|1       |DEPORTES      |María Fernández|maría.fernández@gmail.com |2024-02-08|25.0   |Colchoneta Yoga   |
|4       |  hogar       |Lucía Martínez |lucía.martínez@yahoo.com  |2024-10-14|28.0   |Set de Cubiertos  |
|3       |deportes      |Martín Vega    |martín.vega@mail.com      |2024-10-11|15.0   |Botella Térmica   |
|1       |deportes      |Morena Acosta  |morena.acosta@gmail.com   |2024-11-20|890.0  |Bicicleta Estática|
|3       |TECNOLOGIA    |María Fernández|maría.fernández@gmail.com |2024-04-17|3400.99|Monitor 27"       |
|1       |  hogar       |Pedro Gómez    |pedro.gómez@gmail.com     |2024-12-02|48.0   |Olla a Presión    |
|5       |tecnologia    |María Fernán

In [7]:
df = df.withColumn(
    "venta_total",
    (col("precio") * col("cantidad")).cast("double")
)
df.show(truncate=False)

+--------+--------------+---------------+--------------------------+----------+-------+------------------+------------------+
|cantidad|categoria     |cliente        |email                     |fecha     |precio |producto          |venta_total       |
+--------+--------------+---------------+--------------------------+----------+-------+------------------+------------------+
|1       |DEPORTES      |María Fernández|maría.fernández@gmail.com |2024-02-08|25.0   |Colchoneta Yoga   |25.0              |
|4       |  hogar       |Lucía Martínez |lucía.martínez@yahoo.com  |2024-10-14|28.0   |Set de Cubiertos  |112.0             |
|3       |deportes      |Martín Vega    |martín.vega@mail.com      |2024-10-11|15.0   |Botella Térmica   |45.0              |
|1       |deportes      |Morena Acosta  |morena.acosta@gmail.com   |2024-11-20|890.0  |Bicicleta Estática|890.0             |
|3       |TECNOLOGIA    |María Fernández|maría.fernández@gmail.com |2024-04-17|3400.99|Monitor 27"       |10202.97    

In [23]:
from pyspark.sql.functions import avg

df_promedio_ventas_trimestre = (
    df.groupBy("trimestre")
    .agg(
        avg("venta_total").alias("promedio_venta_trimestre")
    )
)

df_promedio_ventas_trimestre.show(truncate=False)

+---------+------------------------+
|trimestre|promedio_venta_trimestre|
+---------+------------------------+
|1        |1200.33                 |
|3        |467.3333333333333       |
|4        |204.5181818181818       |
|2        |2534.9054545454546      |
+---------+------------------------+



In [19]:
from pyspark.sql.functions import ltrim, col, initcap

df = df.withColumn(
    "categoria",
    initcap(ltrim(col("categoria")))
)

df.show(truncate=False)

+---+--------+------------+---------------+--------------------------+------------+--------+-----------------+
|_id|cantidad|categoria   |cliente        |email                     |fecha       |precio  |producto         |
+---+--------+------------+---------------+--------------------------+------------+--------+-----------------+
|1  |1       |Deportes    |María Fernández|maría.fernández@gmail.com |24/01/2024  |NULL    |Raqueta de Tenis |
|2  |3       |Tecnologia  |María Fernández|maría.fernández@gmail.com |Apr 17, 2024|3.400,99|Monitor 27"      |
|3  |3       |Tecnologia  |María Fernández|maría.fernández@gmail.com |14/12/2024  |NULL    |Teclado Mecánico |
|4  |5       |Hogar       |Agustín Flores |agustín.flores@gmail.com  |2024-05-26  |NULL    |Espejo de Pared  |
|5  |1       |Hogar       |María Fernández|maría.fernández@gmail.com |2024-04-23  |42,00   |Espejo de Pared  |
|6  |1       |Hogar       |Agustín Flores |agustín.flores@gmail.com  |21/08/2024  |140,00  |Aspiradora       |
|